In [32]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
import pandas as pd

In [33]:
HVFHV_PATH = "hdfs:///tlc/raw/hvfhv"
ZONES_PART_PATH = "hdfs:///tlc/raw/zones/part-00000-f8135e00-968c-4f34-a8fc-7e4182eaa810-c000.snappy.parquet"
ZONES_TAXI_PATH = "hdfs:///tlc/raw/zones/taxi_zones.parquet"


In [34]:
DEV_MODE = True
DEV_SAMPLE_SIZE = 500_000

In [39]:
def get_spark() -> SparkSession:
    return (
        SparkSession.builder
        .appName("TLC-EDA")
        .master("local[*]")
        .getOrCreate()
    )
spark = get_spark()
spark.sparkContext.setLogLevel("ERROR")

In [36]:
def inspect_dataframe(df, name):
    print(f"DATASET: {name}")

    # Schema
    print("\nSCHEMA")
    df.printSchema()

    # Columns
    print("\nCOLUMNS")
    for i, column in enumerate(df.columns, 1):
        print(f"{i}. {column}")

    # Row count
    print("\nROW COUNT")
    print(df.count())

    # Sample data
    print("\nSAMPLE DATA")
    sample_pdf = df.limit(10).toPandas()
    display(sample_pdf) 

    # Null counts
    print("\nNULL COUNTS")
    null_counts = df.select([
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in df.columns
    ])
    null_pdf = null_counts.toPandas()
    display(null_pdf)

    # Distinct counts
    print("\nDISTINCT COUNTS")
    distinct_counts = {col: df.select(col).distinct().count() for col in df.columns}
    distinct_pdf = pd.DataFrame(list(distinct_counts.items()), columns=["Column", "DistinctCount"])
    display(distinct_pdf)

    # Statistics
    print("\nSTATISTICS")
    stats_pdf = df.describe().toPandas()
    display(stats_pdf)


In [37]:
print("\nReading HVFHV data...")
hvfhv = spark.read.parquet(HVFHV_PATH)

if DEV_MODE:
    hvfhv = hvfhv.limit(DEV_SAMPLE_SIZE)
    
inspect_dataframe(hvfhv, "HVFHV TRIPS")


Reading HVFHV data...
DATASET: HVFHV TRIPS

SCHEMA
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp (nullable = true)
 |-- on_scene_datetime: timestamp (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: str

,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B03404,B03404,2024-01-01 09:21:47,2024-01-01 09:25:06,2024-01-01 09:28:08,2024-01-01 10:05:39,161,158,2.83,...,4.05,2.75,0.0,0.0,40.18,N,N,N,N,N
1,HV0003,B03404,B03404,2024-01-01 09:10:56,2024-01-01 09:11:08,2024-01-01 09:12:53,2024-01-01 09:20:05,137,79,1.57,...,0.89,2.75,0.0,0.0,6.12,N,N,N,N,N
2,HV0003,B03404,B03404,2024-01-01 09:20:04,2024-01-01 09:21:51,2024-01-01 09:23:05,2024-01-01 09:35:16,79,186,1.98,...,1.60,2.75,0.0,0.0,9.47,N,N,N,N,N
3,HV0003,B03404,B03404,2024-01-01 09:35:46,2024-01-01 09:39:59,2024-01-01 09:41:04,2024-01-01 09:56:34,234,148,1.99,...,1.52,2.75,0.0,0.0,11.35,N,N,N,N,N
4,HV0003,B03404,B03404,2024-01-01 09:48:19,2024-01-01 09:56:23,2024-01-01 09:57:21,2024-01-01 10:10:02,148,97,2.65,...,3.43,2.75,0.0,0.0,28.63,N,N,N,N,N
5,HV0003,B03404,B03404,2024-01-01 09:03:47,2024-01-01 09:05:53,2024-01-01 09:06:15,2024-01-01 09:27:53,255,95,7.02,...,2.85,0.00,0.0,0.0,24.35,N,N,N,N,Y
6,HV0003,B03404,B03404,2024-01-01 09:22:51,2024-01-01 09:29:17,2024-01-01 09:29:47,2024-01-01 09:50:08,95,212,11.33,...,4.68,0.00,0.0,0.0,30.98,N,N,N,N,Y
7,HV0003,B03404,B03404,2024-01-01 09:45:34,2024-01-01 09:57:29,2024-01-01 09:57:50,2024-01-01 10:11:27,213,47,3.43,...,2.06,0.00,0.0,0.0,20.73,N,N,N,N,Y
8,HV0003,B03404,B03404,2024-01-01 09:11:51,2024-01-01 09:15:46,2024-01-01 09:16:00,2024-01-01 09:28:13,209,114,1.54,...,1.37,2.75,0.0,0.0,10.40,N,N,N,N,Y
9,HV0003,B03404,B03404,2024-01-01 09:26:48,2024-01-01 09:33:02,2024-01-01 09:33:15,2024-01-01 09:46:39,113,209,1.72,...,1.21,2.75,0.0,0.0,11.38,N,N,N,N,Y



NULL COUNTS


,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,0,0,156590,0,156590,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0



DISTINCT COUNTS


,Column,DistinctCount
0,hvfhs_license_num,2
1,dispatching_base_num,2
2,originating_base_num,3
3,request_datetime,64598
4,on_scene_datetime,63128
5,pickup_datetime,64377
6,dropoff_datetime,66113
7,PULocationID,258
8,DOLocationID,260
9,trip_miles,23716



STATISTICS


,summary,hvfhs_license_num,dispatching_base_num,originating_base_num,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tolls,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,count,500000,500000,343410,500000,500000,500000,500000,500000,500000,...,500000,500000,500000,500000,500000,500000,500000,500000,500000,500000
1,mean,None,None,None,134.89921,139.412112,5.596106484000043,1089.72017,29.84468240000014,1.3676083999993323,...,2.459829320000606,0.9230165,0.178105,1.0769811600000696,22.925207900010108,None,None,None,None,None
2,stddev,None,None,None,76.02972837262433,79.76200884917297,6.029703509933562,714.8053792121594,24.032557249189445,4.435322345942598,...,1.9328565644499807,1.2917705884057706,0.6460384614760899,3.3795700895222085,16.816341455603546,None,None,None,None,None
3,min,HV0003,B03404,B03404,2,1,0.0,1,-9.57,0.0,...,0.0,0.0,0.0,0.0,-29.98,N,N,N,N,N
4,max,HV0005,B03406,B03406,265,265,227.6,32766,1627.4,64.2,...,111.69,5.5,5.0,100.0,760.42,Y,Y,Y,Y,Y


In [41]:
print("\nReading taxi zone data...")
zones = spark.read.parquet(ZONES_PART_PATH)

inspect_dataframe(zones, "TAXI ZONES")


Reading taxi zone data...
DATASET: TAXI ZONES

SCHEMA
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)


COLUMNS
1. LocationID
2. Borough
3. Zone
4. service_zone

ROW COUNT
265

SAMPLE DATA


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
6,7,Queens,Astoria,Boro Zone
7,8,Queens,Astoria Park,Boro Zone
8,9,Queens,Auburndale,Boro Zone
9,10,Queens,Baisley Park,Boro Zone



NULL COUNTS


,LocationID,Borough,Zone,service_zone
0,0,0,0,0



DISTINCT COUNTS


,Column,DistinctCount
0,LocationID,265
1,Borough,8
2,Zone,262
3,service_zone,5



STATISTICS


,summary,LocationID,Borough,Zone,service_zone
0,count,265,265,265,265
1,mean,133.0,None,None,None
2,stddev,76.643112323722,None,None,None
3,min,1,Bronx,Allerton/Pelham Gardens,Airports
4,max,265,Unknown,Yorkville West,Yellow Zone


In [29]:
print("\nReading shapefile data...")
shapefile = spark.read.parquet(ZONES_TAXI_PATH)

inspect_dataframe(shapefile, "SHAPEFILE")


Reading shapefile data...
DATASET: SHAPEFILE

SCHEMA
root
 |-- geometry: binary (nullable = true)
 |-- OBJECTID: integer (nullable = true)
 |-- Shape_Leng: double (nullable = true)
 |-- Shape_Area: double (nullable = true)
 |-- zone: string (nullable = true)
 |-- LocationID: integer (nullable = true)
 |-- borough: string (nullable = true)


COLUMNS
1. geometry
2. OBJECTID
3. Shape_Leng
4. Shape_Area
5. zone
6. LocationID
7. borough

ROW COUNT
263

SAMPLE DATA


,geometry,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough
0,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 232, 0, 0, 0, 146,...",1,0.116357,0.000782,Newark Airport,1,EWR
1,"[1, 6, 0, 0, 0, 33, 0, 0, 0, 1, 3, 0, 0, 0, 1,...",2,0.433470,0.004866,Jamaica Bay,2,Queens
2,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 121, 0, 0, 0, 0, 4...",3,0.084341,0.000314,Allerton/Pelham Gardens,3,Bronx
3,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 88, 0, 0, 0, 128, ...",4,0.043567,0.000112,Alphabet City,4,Manhattan
4,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 170, 0, 0, 0, 0, 2...",5,0.092146,0.000498,Arden Heights,5,Staten Island
5,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 21, 1, 0, 0, 128, ...",6,0.150491,0.000606,Arrochar/Fort Wadsworth,6,Staten Island
6,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 182, 0, 0, 0, 128,...",7,0.107417,0.000390,Astoria,7,Queens
7,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 40, 0, 0, 0, 0, 20...",8,0.027591,0.000027,Astoria Park,8,Queens
8,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 189, 0, 0, 0, 128,...",9,0.099784,0.000338,Auburndale,9,Queens
9,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 157, 0, 0, 0, 128,...",10,0.099839,0.000436,Baisley Park,10,Queens



NULL COUNTS


,geometry,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough
0,0,0,0,0,0,0,0



DISTINCT COUNTS


,Column,DistinctCount
0,geometry,263
1,OBJECTID,263
2,Shape_Leng,263
3,Shape_Area,263
4,zone,260
5,LocationID,263
6,borough,6



STATISTICS


,summary,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough
0,count,263,263,263,263,263,263
1,mean,132.0,0.094269002579273,4.0182942824071153E-4,None,132.0,None
2,stddev,76.06576102294646,0.054593642765557955,4.822876754039383E-4,None,76.06576102294646,None
3,min,1,0.0143055167343,6.330563613E-6,Allerton/Pelham Gardens,1,Bronx
4,max,263,0.43346966679,0.00486634037837,Yorkville West,263,Staten Island


In [38]:
print("HVFHV DUPLICATES")

total_hvfhv = hvfhv.count()
distinct_hvfhv = hvfhv.dropDuplicates().count()

print(f"Total rows: {total_hvfhv}")
print(f"Distinct rows: {distinct_hvfhv}")
print(f"Duplicate rows: {total_hvfhv - distinct_hvfhv}")


HVFHV DUPLICATES


Total rows: 500000
Distinct rows: 500000
Duplicate rows: 0


In [42]:
print("ZONE DUPLICATES")

total_zones = zones.count()
distinct_zones = zones.dropDuplicates().count()

print(f"Total rows: {total_zones}")
print(f"Distinct rows: {distinct_zones}")
print(f"Duplicate rows: {total_zones - distinct_zones}")

ZONE DUPLICATES
Total rows: 265
Distinct rows: 265
Duplicate rows: 0


In [19]:
# Check duplicate LocationIDs
zones.groupBy("LocationID") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+----------+-----+
|LocationID|count|
+----------+-----+
+----------+-----+



In [17]:
pickup_missing = hvfhv.join(
    zones,
    hvfhv.PULocationID == zones.LocationID,
    "left_anti"
)

print(
    "Pickup LocationIDs without matching zone:",
    pickup_missing.count()
)

dropoff_missing = hvfhv.join(
    zones,
    hvfhv.DOLocationID == zones.LocationID,
    "left_anti"
)
print(
    "Dropoff LocationIDs without matching zone:",
    dropoff_missing.count()
)

Pickup LocationIDs without matching zone: 0
Dropoff LocationIDs without matching zone: 0


In [ ]:
# Check Dates min and max values
timestamp_columns = [
    "request_datetime",
    "on_scene_datetime",
    "pickup_datetime",
    "dropoff_datetime"
]

for column in timestamp_columns:

    if column not in trips.columns:
        print(f"{column}: NOT FOUND")
        continue

    print("\n", "=" * 70)
    print(column)
    print("=" * 70)

    trips.select(
        F.min(column).alias("minimum"),
        F.max(column).alias("maximum"),
        F.count(F.when(F.col(column).isNull(), True))
        .alias("null_count")
    ).show()

# Check dates violation   
   if (
    "pickup_datetime" in trips.columns
    and "dropoff_datetime" in trips.columns
):

    invalid_time = trips.filter(
        F.col("dropoff_datetime") < F.col("pickup_datetime")
    )

    print(
        "Rows where dropoff occurs before pickup:",
        invalid_time.count()
    )

    invalid_time.show(10, truncate=False)

In [31]:
spark.stop()